# Feature Reduction Study

**[CTO] research notebook -- not part of the production serving path.**

This notebook does **not** modify, retrain, or replace the frozen production
Random Forest (`models/random_forest.joblib`). It trains two *separate,
research-only* copies of the same Random Forest architecture under
identical conditions (same rows, same split, same preprocessing, same
hyperparameters, same `random_state`) to answer one question:

> If an agent only enters 7 fields (Policy Tenure, Driver Age, Vehicle Age,
> Fuel Type, Transmission Type, NCAP Rating, Airbags), how much predictive
> performance is lost compared to the full 39-field contract?

- **Model A** -- the full production feature set (61 encoded columns, the
  same set used by the frozen `models/random_forest.joblib`).
- **Model B** -- only the 10 encoded columns derived from the 7 raw fields
  listed above (`fuel_type` and `transmission_type` are one-hot encoded).

Both models reuse the exact preprocessed CSVs and split produced in Sprint 3
(`data/processed/sprint_03_preprocessed_v1/`) and the exact hyperparameters
frozen for production in `ml_pipeline/ml_engineer.py`
(`FINAL_RANDOM_FOREST_CONFIG`). This makes Model A and Model B directly
comparable to each other, and Model A directly comparable to the frozen
production model as a reproducibility check.

In [1]:
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

PROCESSED_DIR = "../data/processed/sprint_03_preprocessed_v1/"

X_train = pd.read_csv(PROCESSED_DIR + "X_train_model_ready.csv")
y_train = pd.read_csv(PROCESSED_DIR + "y_train.csv")["is_claim"]
X_val = pd.read_csv(PROCESSED_DIR + "X_validation_model_ready.csv")
y_val = pd.read_csv(PROCESSED_DIR + "y_validation.csv")["is_claim"]
X_hold = pd.read_csv(PROCESSED_DIR + "X_holdout_model_ready.csv")
y_hold = pd.read_csv(PROCESSED_DIR + "y_holdout.csv")["is_claim"]

# Same development set the frozen production model was trained on
# (train + validation combined; holdout is reserved for evaluation only).
X_dev = pd.concat([X_train, X_val], axis=0, ignore_index=True)
y_dev = pd.concat([y_train, y_val], axis=0, ignore_index=True)

FULL_FEATURES = list(X_dev.columns)

# The 7 raw Agent Mode fields, encoded the same way the frozen preprocessing
# pipeline encodes them (fuel_type / transmission_type -> one-hot; the rest
# pass through as scaled/raw numeric columns already present in FULL_FEATURES).
MODEL_B_FEATURES = [
    "policy_tenure",
    "age_of_car",
    "age_of_policyholder",
    "airbags",
    "ncap_rating",
    "fuel_type__CNG",
    "fuel_type__Diesel",
    "fuel_type__Petrol",
    "transmission_type__Automatic",
    "transmission_type__Manual",
]
assert all(f in FULL_FEATURES for f in MODEL_B_FEATURES)

print(f"Development rows (train+validation): {len(X_dev)}")
print(f"Holdout rows: {len(X_hold)}")
print(f"Full feature count (Model A): {len(FULL_FEATURES)}")
print(f"Model B feature count (encoded): {len(MODEL_B_FEATURES)}")

Development rows (train+validation): 49804
Holdout rows: 8788
Full feature count (Model A): 61
Model B feature count (encoded): 10


In [2]:
# Exact frozen production hyperparameters from ml_pipeline/ml_engineer.py
# (FINAL_RANDOM_FOREST_CONFIG). Reused unmodified for both Model A and
# Model B so the only experimental variable is the feature set.
RF_CONFIG = dict(
    n_estimators=200,
    max_depth=8,
    min_samples_split=200,
    min_samples_leaf=50,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42,
)


def evaluate(model, X, y, threshold=0.5):
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "roc_auc": roc_auc_score(y, proba),
        "pr_auc": average_precision_score(y, proba),
    }

## Model A -- full production feature set (61 columns)

In [3]:
model_a = RandomForestClassifier(**RF_CONFIG)
model_a.fit(X_dev[FULL_FEATURES], y_dev)
metrics_a = evaluate(model_a, X_hold[FULL_FEATURES], y_hold)
print(json.dumps(metrics_a, indent=2))

{
  "accuracy": 0.5978607191624943,
  "precision": 0.0988120950323974,
  "recall": 0.6512455516014235,
  "f1": 0.17158931082981715,
  "roc_auc": 0.6619938473012832,
  "pr_auc": 0.11210550241803961
}


These numbers match the frozen production model's official holdout
evaluation in `models/random_forest_metadata.json` (accuracy 0.5979,
precision 0.0988, recall 0.6512, F1 0.1716, ROC-AUC 0.6620, PR-AUC 0.1109)
to within floating-point/tie-breaking noise on PR-AUC. This confirms the
research setup in this notebook faithfully reproduces the frozen training
conditions, so the Model A vs. Model B comparison below is apples-to-apples.

## Model B -- 7-field Agent Mode subset (10 encoded columns)

In [4]:
model_b = RandomForestClassifier(**RF_CONFIG)
model_b.fit(X_dev[MODEL_B_FEATURES], y_dev)
metrics_b = evaluate(model_b, X_hold[MODEL_B_FEATURES], y_hold)
print(json.dumps(metrics_b, indent=2))

{
  "accuracy": 0.5757851615839782,
  "precision": 0.0974059003051882,
  "recall": 0.6814946619217082,
  "f1": 0.17044948820649755,
  "roc_auc": 0.6622383848452048,
  "pr_auc": 0.10856520907582316
}


## Side-by-side comparison

In [5]:
comparison = pd.DataFrame({"Model A (61 features)": metrics_a, "Model B (7 fields)": metrics_b})
comparison["Delta (A - B)"] = comparison["Model A (61 features)"] - comparison["Model B (7 fields)"]
comparison

           Model A (61 features)  Model B (7 fields)  Delta (A - B)
accuracy                0.597861             0.575785       0.022076
precision                0.098812             0.097406       0.001406
recall                   0.651246             0.681495      -0.030249
f1                        0.171589             0.170449       0.001140
roc_auc                   0.661994             0.662238      -0.000244
pr_auc                    0.112106             0.108565       0.003540

## Statistical significance (bootstrap on the holdout set)

1,000 bootstrap resamples of the 8,788-row holdout set, recomputing the
ROC-AUC and F1 difference between Model A and Model B on each resample, to
get a 95% confidence interval for the true difference.

In [6]:
proba_a = model_a.predict_proba(X_hold[FULL_FEATURES])[:, 1]
proba_b = model_b.predict_proba(X_hold[MODEL_B_FEATURES])[:, 1]
y_hold_arr = y_hold.to_numpy()

rng = np.random.RandomState(42)
n = len(y_hold_arr)
diffs_auc, diffs_f1 = [], []
for _ in range(1000):
    idx = rng.randint(0, n, n)
    yb = y_hold_arr[idx]
    if yb.sum() == 0 or yb.sum() == n:
        continue
    pa, pb = proba_a[idx], proba_b[idx]
    diffs_auc.append(roc_auc_score(yb, pa) - roc_auc_score(yb, pb))
    diffs_f1.append(
        f1_score(yb, (pa >= 0.5).astype(int)) - f1_score(yb, (pb >= 0.5).astype(int))
    )

diffs_auc, diffs_f1 = np.array(diffs_auc), np.array(diffs_f1)
auc_ci = np.percentile(diffs_auc, [2.5, 97.5])
f1_ci = np.percentile(diffs_f1, [2.5, 97.5])
print(f"ROC-AUC diff (A-B): mean {diffs_auc.mean():.6f}, 95% CI [{auc_ci[0]:.6f}, {auc_ci[1]:.6f}]")
print(f"F1 diff (A-B): mean {diffs_f1.mean():.6f}, 95% CI [{f1_ci[0]:.6f}, {f1_ci[1]:.6f}]")

ROC-AUC diff (A-B): mean -0.000278, 95% CI [-0.010459, 0.009410]
F1 diff (A-B): mean 0.001193, 95% CI [-0.006163, 0.008849]


Both 95% confidence intervals contain 0. The difference between Model A and
Model B on ROC-AUC and F1 is **not statistically significant** at this
sample size -- see `docs/reports/feature_reduction_study.md` Phase 4 for the
business interpretation.

## Phase 3 -- Feature Importance Validation (frozen production model)

This section does **not** use the research Model A/B above. It loads the
actual frozen `models/random_forest.joblib` and asks: of the 7 fields
selected for Agent Mode, are they really the model's most influential
features? We check two independent importance measures: the model's
built-in (Gini/impurity-based) `feature_importances_`, and permutation
importance (ROC-AUC drop when a column is shuffled) computed directly on
the holdout set.

In [7]:
production_model = joblib.load("../models/random_forest.joblib")
production_feature_names = json.load(open("../models/random_forest_metadata.json"))["feature_names"]

gini_importance = pd.Series(
    production_model.feature_importances_, index=production_feature_names
).sort_values(ascending=False)

print("Top 10 features by built-in (Gini) importance, frozen production model:")
for rank, (name, value) in enumerate(gini_importance.head(10).items(), start=1):
    print(f"{rank:2d}. {name:<30} {value:.5f}")

Top 10 features by built-in (Gini) importance, frozen production model:
 1. policy_tenure                 0.36997
 2. age_of_car                    0.27204
 3. age_of_policyholder           0.09760
 4. area_cluster__freq            0.07191
 5. population_density            0.07021
 6. power_to_weight                0.00783
 7. model__freq                    0.00678
 8. torque_nm                      0.00658
 9. vehicle_volume_proxy           0.00654
10. height                         0.00646


In [8]:
permutation_result = permutation_importance(
    production_model,
    X_hold[production_feature_names],
    y_hold,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
)
permutation_importance_series = pd.Series(
    permutation_result.importances_mean, index=production_feature_names
).sort_values(ascending=False)

print("Top 10 features by permutation importance (ROC-AUC drop), frozen production model:")
for rank, (name, value) in enumerate(permutation_importance_series.head(10).items(), start=1):
    print(f"{rank:2d}. {name:<30} {value:.5f}")

Top 10 features by permutation importance (ROC-AUC drop), frozen production model:
 1. policy_tenure                 0.07938
 2. age_of_car                    0.07372
 3. age_of_policyholder           0.00527
 4. area_cluster__freq            0.00302
 5. population_density            0.00103
 6. torque_nm                      0.00101
 7. power_to_weight                0.00073
 8. power_bhp                      0.00073
 9. vehicle_volume_proxy           0.00072
10. length                         0.00043


In [9]:
rank_lookup = {name: i + 1 for i, name in enumerate(gini_importance.index)}

agent_mode_encoded_columns = MODEL_B_FEATURES  # same 10 columns as Model B
print("Gini-importance rank of each Agent Mode (7-field) encoded column, out of 61:")
for name in agent_mode_encoded_columns:
    print(f"  {name:<28} rank {rank_lookup[name]:>2}   importance {gini_importance[name]:.5f}")

print("\nNOT in the 7-field Agent Mode set, but ranked above 4 of its encoded columns:")
for name in ["area_cluster__freq", "population_density"]:
    print(f"  {name:<28} rank {rank_lookup[name]:>2}   importance {gini_importance[name]:.5f}")

print(f"\nSum of importance for the top 5 features overall: {gini_importance.head(5).sum():.4f}")

Gini-importance rank of each Agent Mode (7-field) encoded column, out of 61:
  policy_tenure                  rank  1   importance 0.36997
  age_of_policyholder            rank  3   importance 0.09760
  age_of_car                     rank  2   importance 0.27204
  ncap_rating                    rank 22   importance 0.00288
  airbags                        rank 41   importance 0.00059
  fuel_type__CNG                 rank 34   importance 0.00073
  fuel_type__Diesel              rank 46   importance 0.00047
  fuel_type__Petrol               rank 33   importance 0.00082
  transmission_type__Automatic    rank 39   importance 0.00063
  transmission_type__Manual       rank 30   importance 0.00091

NOT in the 7-field Agent Mode set, but ranked above 4 of its encoded columns:
  area_cluster__freq             rank  4   importance 0.07191
  population_density             rank  5   importance 0.07021

Sum of importance for the top 5 features overall: 0.8817


### Finding

The hypothesis that the 7 selected Agent Mode fields are "the most
influential features" is **only partially true, and verified -- not
assumed -- false in detail**:

- 3 of the 7 (`policy_tenure`, `age_of_car`, `age_of_policyholder`) are
  genuinely the model's #1, #2, and #3 most important features by both
  measures, together carrying ~74% of total Gini importance.
- The other 4 (`ncap_rating`, `airbags`, `fuel_type`, `transmission_type`,
  10 encoded columns) rank between 22nd and 46th out of 61 -- each
  contributing well under 0.3% importance individually.
- Two fields **not** in the 7-field set -- `area_cluster` (rank 4, 7.2%)
  and `population_density` (rank 5, 7.0%) -- are individually more
  important than any of those 4 weak fields.

In other words: the Agent Mode field list is a good UX choice (it picks the
3 dominant drivers), but it is not the evidence-optimal 7-feature subset --
`area_cluster` and `population_density` would be better picks than
`fuel_type`/`transmission_type`/`ncap_rating`/`airbags` if pure predictive
power were the only goal. See `docs/reports/feature_reduction_study.md` for
why this still doesn't change the final recommendation.